In [0]:
def _research_deps_ok():
    try:
        import langchain_community  # noqa: F401
        import databricks_langchain  # noqa: F401
        import wikipedia  # noqa: F401
        import ddgs  # noqa: F401
        import youtube_search  # noqa: F401
        from mcp.shared.context import RequestContext  # noqa: F401
        return True
    except Exception:
        return False


if not _research_deps_ok():
    get_ipython().system(
        "uv pip install langchain-community wikipedia ddgs youtube-search databricks-langchain 'mcp>=1,<2'"
    )
    dbutils.library.restartPython()
    print(
        "Installed dependencies and restarted Python. "
        "Rerun the notebook from this cell to continue."
    )
else:
    print("Research agent dependencies already present - skipping install.")

Using Python 3.12.3 environment at: /local_disk0/.ephemeral_nfs/envs/pythonEnv-61541caf-1d2a-4a71-9340-ea4e6c83b9e5
Resolved 158 packages in 1.06s
Prepared 1 package in 72ms
Uninstalled 1 package in 5ms
Installed 1 package in 11ms
 - mcp==2.3.0
 + mcp==1.30.0
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Update this to an endpoint available in YOUR Databricks workspace.
RESEARCH_MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"


In [0]:
from databricks_langchain import ChatDatabricks

research_llm = ChatDatabricks(
    endpoint=RESEARCH_MODEL_ENDPOINT,
    temperature=0,
)

In [0]:
import logging

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper, DuckDuckGoSearchAPIWrapper
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_community.tools.youtube.search import YouTubeSearchTool
from langchain_core.tools import Tool

logger = logging.getLogger("research_agent_tools")


def safe_tool(name: str, description: str, func):
    """Wrap a tool so a failure returns a readable message to the agent
    instead of raising an exception that crashes the whole multi-agent graph."""

    def _run(query: str) -> str:
        try:
            return str(func(query))
        except Exception as e:
            logger.warning("Tool '%s' failed on query %r: %s", name, query, e)
            return (
                f"Tool '{name}' failed with {type(e).__name__}: {e}. "
                "This source could not be searched. "
                "Try another available research tool or query. "
                "Do not treat this failure as evidence that information "
                "does not exist."
            )

    return Tool(name=name, description=description, func=_run)


# Wikipedia: Wikimedia rate-limits (403/429) generic bot user agents, which
# previously returned HTML instead of JSON and raised JSONDecodeError.
# The safe wrapper makes any such failure degrade gracefully.
wikipedia_raw = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(top_k_results=3, doc_content_chars_max=2500)
)

# Web search: MUST set region explicitly. ddgs's default region "wt-wt" makes
# its Wikipedia fallback engine build https://wt.wikipedia.org, a domain that
# does not exist — this was the DNSError that failed the evaluation traces.
# DuckDuckGoSearchResults (rather than DuckDuckGoSearchRun) is used so results
# include source links, which the system prompt requires the agent to cite.
web_search_raw = DuckDuckGoSearchResults(
    api_wrapper=DuckDuckGoSearchAPIWrapper(region="us-en"),
    output_format="list",
)


def _format_web_results(query: str) -> str:
    results = web_search_raw.invoke(query)
    if not results:
        return "No web results found."
    lines = []
    for r in results:
        title = r.get("title", "")
        link = r.get("link", "")
        snippet = r.get("snippet", "")
        lines.append(f"{title} — {link}\n{snippet}")
    return "\n\n".join(lines)


youtube_search_raw = YouTubeSearchTool()

# Keep the same tool names ("wikipedia", "duckduckgo_search", "youtube_search")
# so the supervisor prompts, agent routing, and eval scorers keep matching.
research_tools = [
    safe_tool(
        "wikipedia",
        "Search Wikipedia for encyclopedic background on people, places, "
        "companies, and historical events. Input should be a search query.",
        wikipedia_raw.invoke,
    ),
    safe_tool(
        "duckduckgo_search",
        "Search the web for current public information. Returns titles, "
        "URLs, and snippets. Input should be a search query.",
        _format_web_results,
    ),
    safe_tool(
        "youtube_search",
        "Search YouTube for videos. Input should be a search query.",
        youtube_search_raw.invoke,
    ),
]

/home/spark-61541caf-1d2a-4a71-9340-ea/.ipykernel/79/command-8256429562993301-3824261392:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [0]:
from langchain.agents import create_agent


SYSTEM_PROMPT = """
You are a reliable research assistant responsible for gathering
accurate, publicly available information.

RESEARCH PROCESS:

1. Use external research tools for factual, current, or
   source-specific questions.

2. Start with the exact name or subject provided by the user.

3. If the initial search does not return useful results:
   - Try alternate names and spellings.
   - Try shorter and more specific search queries.
   - Search for the organization's official website.
   - Search for relevant official announcements and reports.

4. Prioritize sources in this order:
   - Official organization websites.
   - Government and regulatory websites.
   - Recognized news organizations.
   - Other publicly accessible sources.
   - Wikipedia and YouTube as supplementary sources.

5. Do not depend on a single search tool.
   If one tool fails or returns irrelevant results,
   try another available tool.

6. Never interpret a tool failure as proof that information
   does not exist.

7. Clearly distinguish:
   - Verified information.
   - Information reported by external sources.
   - Unverified information.
   - Information that could not be found.

8. Do not invent facts, citations, URLs, or search results.

9. Provide source names and URLs whenever the tool results
   make them available.

10. If sufficient evidence cannot be found after reasonable
    search attempts, explain exactly what was unsuccessful
    and what remains uncertain.

FINAL RESPONSE:

Provide a structured summary containing:
- Background
- Important findings
- Relevant services or activities
- Sources consulted
- Limitations and uncertainties

Be concise, factual, and transparent.
"""


research_agent = create_agent(
    model=research_llm,
    tools=research_tools,
    system_prompt=SYSTEM_PROMPT,
)

